# STAT 8017B Project 4 — Regression & Prediction Models
## Financial Analysis Chatbot (Group 4.1)

This notebook trains regression models for two prediction tasks:

- **Task A**: Predict fund returns from fund characteristics (ETF + Mutual Fund data)
- **Task B**: Predict next-day stock returns from lagged price features

**Models**: Linear Regression, Random Forest Regressor

**Course methods**: Regression (Ch.5), Random Forest (Ch.4), Feature importance

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')

PROCESSED_DIR = os.path.join('..', 'data', 'processed')
MODEL_DIR = os.path.join('..', 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

print('Setup complete.')

In [ ]:
def evaluate_regression(name, model, X_test, y_test):
    """Evaluate a regression model and print metrics."""
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print(f'\n{"=" * 50}')
    print(f'{name}')
    print(f'{"=" * 50}')
    print(f'R-squared: {r2:.4f}')
    print(f'MAE:       {mae:.4f}')
    print(f'RMSE:      {rmse:.4f}')

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].scatter(y_test, y_pred, alpha=0.3, s=10)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    axes[0].plot(lims, lims, 'r--', alpha=0.7)
    axes[0].set_xlabel('Actual')
    axes[0].set_ylabel('Predicted')
    axes[0].set_title(f'{name} — Actual vs Predicted')

    residuals = y_test - y_pred
    axes[1].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
    axes[1].axvline(0, color='red', linestyle='--')
    axes[1].set_xlabel('Residual')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title(f'{name} — Residual Distribution')

    plt.tight_layout()
    plt.show()

    return {'model': name, 'R2': r2, 'MAE': mae, 'RMSE': rmse}

print('Evaluation function ready.')

---
## Task A: Fund Return Prediction

Predict 1-year return from expense ratio, fund size, yield, volatility, beta, and other characteristics.

In [ ]:
etf = pd.read_csv(os.path.join(PROCESSED_DIR, 'etf_clean.csv'))
mf = pd.read_csv(os.path.join(PROCESSED_DIR, 'mutualfund_clean.csv'))

etf['source'] = 'ETF'
mf['source'] = 'MutualFund'

common_cols = list(set(etf.columns) & set(mf.columns))
funds = pd.concat([etf[common_cols], mf[common_cols]], ignore_index=True)
print(f'Combined funds: {len(funds):,} rows')
print(f'Columns: {common_cols}')

In [ ]:
TARGET = 'fund_return_1year'

feature_cols = [
    'fund_annual_report_net_expense_ratio', 'total_net_assets', 'fund_yield',
    'fund_return_ytd', 'fund_return_3years', 'fund_return_5years',
    'fund_mean_annual_return_3years', 'fund_stdev_3years',
    'fund_sharpe_ratio_3years', 'fund_beta_3years',
    'asset_stocks', 'asset_bonds'
]
feature_cols = [c for c in feature_cols if c in funds.columns]

fund_model_df = funds[feature_cols + [TARGET]].dropna()
print(f'Rows with complete data: {len(fund_model_df):,}')

X_fund = fund_model_df[feature_cols].values
y_fund = fund_model_df[TARGET].values

scaler_fund = StandardScaler()
X_fund_scaled = scaler_fund.fit_transform(X_fund)

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fund_scaled, y_fund, test_size=0.2, random_state=42
)
print(f'Train: {X_train_f.shape[0]:,}  |  Test: {X_test_f.shape[0]:,}')
print(f'Features: {feature_cols}')

### A1. Linear Regression

In [ ]:
lr_fund = LinearRegression()
lr_fund.fit(X_train_f, y_train_f)
res_lr_fund = evaluate_regression('Linear Regression — Fund Returns', lr_fund, X_test_f, y_test_f)

coef_df = pd.DataFrame({'feature': feature_cols, 'coefficient': lr_fund.coef_})
coef_df = coef_df.reindex(coef_df['coefficient'].abs().sort_values(ascending=False).index)
print('\nFeature coefficients (standardized):')
print(coef_df.to_string(index=False))

joblib.dump(lr_fund, os.path.join(MODEL_DIR, 'fund_return_lr.joblib'))
joblib.dump(scaler_fund, os.path.join(MODEL_DIR, 'fund_return_scaler.joblib'))

### A2. Random Forest Regressor

In [ ]:
rf_fund = RandomForestRegressor(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1)
rf_fund.fit(X_train_f, y_train_f)
res_rf_fund = evaluate_regression('Random Forest — Fund Returns', rf_fund, X_test_f, y_test_f)

importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_fund.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(importances['feature'], importances['importance'])
ax.set_xlabel('Feature Importance')
ax.set_title('Random Forest — Feature Importances for Fund Return Prediction')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

joblib.dump(rf_fund, os.path.join(MODEL_DIR, 'fund_return_rf.joblib'))

### Task A Summary

In [ ]:
results_fund = pd.DataFrame([res_lr_fund, res_rf_fund])
print(results_fund.to_string(index=False))

---
## Task B: Stock Price Trend Prediction

Predict next-day return using lagged features: past returns, moving average crossovers.

In [ ]:
prices = pd.read_csv(os.path.join(PROCESSED_DIR, 'prices_clean.csv'))
prices['Date'] = pd.to_datetime(prices['Date'])
prices = prices.sort_values(['ticker', 'Date']).reset_index(drop=True)
print(f'Prices: {len(prices):,} rows, {prices["ticker"].nunique()} tickers')
prices.head()

In [ ]:
price_col = 'Close' if 'Close' in prices.columns else 'Adj Close'

# Lagged features per ticker
for lag in [1, 2, 3, 5]:
    prices[f'return_lag{lag}'] = prices.groupby('ticker')['daily_return'].shift(lag)

prices['ma_cross'] = (prices['ma_20'] - prices['ma_50']) / prices[price_col].clip(lower=0.01)
prices['volatility_5d'] = prices.groupby('ticker')['daily_return'].transform(
    lambda x: x.rolling(5, min_periods=1).std()
)

# Target: next-day return
prices['target_return'] = prices.groupby('ticker')['daily_return'].shift(-1)

price_features = ['return_lag1', 'return_lag2', 'return_lag3', 'return_lag5',
                   'ma_cross', 'volatility_5d']

price_model_df = prices[price_features + ['target_return']].dropna()
print(f'Price model rows: {len(price_model_df):,}')

X_price = price_model_df[price_features].values
y_price = price_model_df['target_return'].values

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_price, y_price, test_size=0.2, random_state=42
)
print(f'Train: {X_train_p.shape[0]:,}  |  Test: {X_test_p.shape[0]:,}')

### B1. Linear Regression — Stock Returns

In [ ]:
lr_price = LinearRegression()
lr_price.fit(X_train_p, y_train_p)
res_lr_price = evaluate_regression('Linear Regression — Stock Returns', lr_price, X_test_p, y_test_p)
joblib.dump(lr_price, os.path.join(MODEL_DIR, 'stock_return_lr.joblib'))

### B2. Random Forest — Stock Returns

In [ ]:
rf_price = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_price.fit(X_train_p, y_train_p)
res_rf_price = evaluate_regression('Random Forest — Stock Returns', rf_price, X_test_p, y_test_p)

importances_p = pd.DataFrame({
    'feature': price_features,
    'importance': rf_price.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(importances_p['feature'], importances_p['importance'])
ax.set_xlabel('Feature Importance')
ax.set_title('Random Forest — Feature Importances for Stock Return Prediction')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

joblib.dump(rf_price, os.path.join(MODEL_DIR, 'stock_return_rf.joblib'))

### Task B Summary

In [ ]:
results_price = pd.DataFrame([res_lr_price, res_rf_price])
print(results_price.to_string(index=False))

---
## All Regression Models Summary

In [ ]:
all_results = pd.DataFrame([res_lr_fund, res_rf_fund, res_lr_price, res_rf_price])
print(all_results.to_string(index=False))

print('\n=== Saved Models ===')
for f in sorted(os.listdir(MODEL_DIR)):
    if 'fund' in f or 'stock' in f:
        size_kb = os.path.getsize(os.path.join(MODEL_DIR, f)) / 1024
        print(f'  {f:40s} {size_kb:8.1f} KB')